In [ ]:
%matplotlib inline
import sys
import os
sys.path.append(os.path.abspath(".."))
import importlib

from functions import utils
importlib.reload(utils)
from functions.utils import *

from functions import config
importlib.reload(config)
from functions.config import *

In [ ]:
meal_df=pd.read_excel(meal_data_path)
metadata=pd.read_excel(metadata_path)
metadata.rename(columns={"ID": "participant_id"}, inplace=True)
metadata=clean_string_column(metadata,string_cols)

In [ ]:
numeric_cols = [
    "carbs_g",
    "protein_g",
    "fat_g",
    "fiber_g",
    "glycemic_index",
    "total_calories"
]

processing_category_meanings = {
    1: "Unprocessed or minimally processed",
    2: "Processed Culinary Ingredients",
    3: "Processed",
    4: "Ultra-Processed"
}

In [ ]:
#Decoding and cleaning of the meal image annotations for the valid (CGM available) participants
#----------------------------------------------------------------------------------------------

#valid_participants are the ones with complete cgm data, 28 and 43 did not have it even in the related column it said 'yes'
completed_cgm_metadata=metadata[metadata['complete_cgm_data']=='yes']
participant_ids=completed_cgm_metadata['participant_id'].tolist()
participant_ids.remove('AL28')
participant_ids.remove('AL43')

valid_participants_meal_events = meal_df[meal_df["participant_id"].isin(participant_ids)]
valid_participants_meal_events=clean_numeric_columns(valid_participants_meal_events, numeric_cols)
valid_participants_meal_events['diet_type']=valid_participants_meal_events['diet_type'].apply(decode_diet)
valid_participants_meal_events['ultra_processed']=valid_participants_meal_events['ultra_processed'].apply(decode_yesno)
valid_participants_meal_events['added_sugar']=valid_participants_meal_events['added_sugar'].apply(decode_yesno)
valid_participants_meal_events['fluid_or_solid']=valid_participants_meal_events['fluid_or_solid'].apply(decode_fluid)
valid_participants_meal_events = valid_participants_meal_events.drop(columns=['diet_type_decision_basis','additional_information', 'confidence_evidence'])


In [ ]:
merged_dataset = valid_participants_meal_events.merge(
        metadata,
        on="participant_id",
        how="left"
    )
merged_dataset = add_time_features(merged_dataset,'datetime')

In [ ]:
merged_dataset.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\results\excels\merged_meal_meta_valid_participants.xlsx')
valid_participants_meal_events.to_excel(r'C:\Users\aoezek\Desktop\master_thesis\master_thesis\finalised_clean_notebooks\results\excels\meal_labels_output_clean_decoded.xlsx')

---
##### Some Visualisations
---

In [ ]:
color_palette= [
    "#1b9e77",  # teal green
    "#d95f02",  # burnt orange
]
def set_plot_style():
    sns.set_palette(color_palette)
    sns.set_style("whitegrid")
    sns.set_context("notebook", font_scale=1.2)
    plt.rcParams["axes.labelsize"] = 12
    plt.rcParams["axes.titlesize"] = 14
    plt.rcParams["legend.fontsize"] = 12
    plt.rcParams["figure.figsize"] = (8, 5)


In [ ]:
def plot_meal_timing_distribution(df,group_col):
    """
    Plot meal timing distribution (of hour of day) by group (e.g. age, sex).
    """
    set_plot_style()
    plt.figure()
    sns.histplot(
        data=df,
        x="hour",
        hue=group_col,
        bins=24,
        element="step",
        palette=color_palette
    )

    plt.xlabel("Hour of Day")
    plt.ylabel("Meal Count")
    plt.title(f"Meal Timing Distribution by {group_col}")
    plt.show()

plot_meal_timing_distribution(merged_dataset, group_col="age_group")
plot_meal_timing_distribution(merged_dataset, group_col="sex")

In [ ]:
def plot_ultra_processed_ratio(df,group_col):
    """
    Plots proportion of ultra-processed meals by group (e.g. age, sex).
    """
    proportion_df = (
        df.groupby(group_col)["ultra_processed"]
        .mean()
        .reset_index()
    )
    set_plot_style()
    plt.figure()
    sns.barplot(
        data=proportion_df,
        x=group_col,
        y="ultra_processed",
        palette=color_palette
    )

    plt.ylabel("Proportion Ultra-Processed")
    plt.title(f"Ultra-Processed Meals by {group_col}")
    plt.show()

plot_ultra_processed_ratio(merged_dataset,"age_group")
plot_ultra_processed_ratio(merged_dataset, "sex")